# Apple Product Pricing & Competitive Intelligence Analysis (2020–2026)

* Analyzed 80K+ daily Apple pricing records (2020–2026) across Amazon and Flipkart to uncover pricing, discount, and stock trends across iPhone, iPad, Mac, and Watch categories.

* Compared platform pricing strategy, festival-sale discount behavior, and price-rating relationships to identify over/underpriced products and competitive gaps.

* Built a full SQL → Python → Power BI pipeline to turn raw pricing data into executive-level business insights.

In [2]:
from sqlalchemy import create_engine, text
import pandas as pd
import os
import numpy as np

In [3]:
engine = create_engine(
    f"mysql+mysqlconnector://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}/{os.environ['DB_NAME']}"
)

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

print("Total Tables:", len(tables))
print("Table Names:")
for table in tables:
    print("-", table)

Total Tables: 1
Table Names:
- apple_products_pricing_2020_2026


In [4]:
for table in tables:
       print(f"\n Table: {table}")
       query = text(f"SELECT * FROM {table}")
       df = pd.read_sql_query(query, engine)
       display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", engine))


 Table: apple_products_pricing_2020_2026


,Date,Platform,Product_Category,Model_Name,Condition,Launch_Price_USD,Launch_Price_INR,Current_Price_USD,Current_Price_INR,Discount_Pct,Sale_Event,Stock_Status,Rating,Reviews_Count
0,2020-09-19,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,435.81,43322.4,-1.6,,In Stock,4.7,40
1,2020-09-20,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.49,42320.4,-1.7,,Out of Stock,4.6,84
2,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,422.73,40879.4,1.5,,In Stock,4.4,110
3,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,425.00,42008.7,0.9,,In Stock,4.8,111
4,2020-09-24,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.22,41984.3,-1.7,,In Stock,4.7,35


In [5]:
df.replace("NaN", np.nan, inplace=True)

In [6]:
print(f"data info : {df.info()}")
print("-"* 40)
print(f"null check : \n{df.isnull().sum()}")
print("-"* 40)
print(f"duplicate data : {df.duplicated().sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               80000 non-null  object 
 1   Platform           80000 non-null  object 
 2   Product_Category   80000 non-null  object 
 3   Model_Name         80000 non-null  object 
 4   Condition          80000 non-null  object 
 5   Launch_Price_USD   80000 non-null  int64  
 6   Launch_Price_INR   80000 non-null  int64  
 7   Current_Price_USD  80000 non-null  float64
 8   Current_Price_INR  80000 non-null  float64
 9   Discount_Pct       80000 non-null  float64
 10  Sale_Event         80000 non-null  object 
 11  Stock_Status       80000 non-null  object 
 12  Rating             80000 non-null  float64
 13  Reviews_Count      80000 non-null  int64  
dtypes: float64(4), int64(3), object(7)
memory usage: 8.5+ MB
data info : None
----------------------------------------
nul

In [7]:
# remove leading and trailing spaces from column names and change lowercase 
df.columns = df.columns.str.strip().str.lower()

In [8]:
# remove whitespace from object columns and
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

In [9]:
# check unique values
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    vals = df[col].unique()
    print(f"\n{col} ({len(vals)} unique): \n{vals} ...")


date (2130 unique): 
['2020-09-19' '2020-09-20' '2020-09-23' ... '2026-07-29' '2026-07-30'
 '2026-07-31'] ...

platform (2 unique): 
['Flipkart' 'Amazon'] ...

product_category (4 unique): 
['Watch' 'iPad' 'iPhone' 'Mac'] ...

model_name (31 unique): 
['Apple Watch Series 6 (44mm)' 'iPad Air (4th Gen) 64GB' 'iPhone 12 64GB'
 'iPhone 12 Pro 128GB' 'MacBook Air M1 256GB'
 'iPad Pro 11-inch (M1) 128GB' 'iPhone 13 128GB' 'iPhone 13 Pro Max 256GB'
 'iPad (9th Gen) 64GB' 'Apple Watch Series 7 (45mm)'
 'MacBook Pro 14-inch M1 Pro 512GB' 'iPad Air (5th Gen) 64GB'
 'MacBook Air M2 256GB' 'Apple Watch Series 8 (45mm)' 'iPhone 14 128GB'
 'iPhone 14 Pro 128GB' 'Apple Watch Ultra' 'iPad Pro 12.9-inch (M2) 256GB'
 'MacBook Pro 14-inch M2 Pro 512GB' 'Apple Watch Series 9 (45mm)'
 'Apple Watch Ultra 2' 'iPhone 15 128GB' 'iPhone 15 Pro Max 256GB'
 'MacBook Pro 14-inch M3 Pro 512GB' 'MacBook Air M3 256GB'
 'iPad Pro 11-inch (M4) 256GB' 'iPhone 16 128GB' 'iPhone 16 Pro 256GB'
 'Apple Watch Series X (45m

In [10]:
df['sale_event'].value_counts()

sale_event
                         73351
Black Friday              2497
Big Billion Days          1579
Great Indian Festival     1504
Prime Day                 1069
Name: count, dtype: int64

👆**here '' means no sales event in particular days so we replace '' to 'no_sales_event'**

In [11]:
df["sale_event"] = (
    df["sale_event"]
      .str.strip()
      .replace("", "No Sale Event")
)

In [12]:
df['sale_event'].value_counts()

sale_event
No Sale Event            73351
Black Friday              2497
Big Billion Days          1579
Great Indian Festival     1504
Prime Day                 1069
Name: count, dtype: int64

In [13]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

In [14]:
df.columns

Index(['date', 'platform', 'product_category', 'model_name', 'condition',
       'launch_price_usd', 'launch_price_inr', 'current_price_usd',
       'current_price_inr', 'discount_pct', 'sale_event', 'stock_status',
       'rating', 'reviews_count', 'year', 'month'],
      dtype='object')

In [16]:
df.to_sql(name="clean_dataset",
          con=engine , 
          if_exists="replace" , 
          index=False,
          chunksize=500)
print("clean dataset save in mysql")

clean dataset save in mysql


In [17]:
df_verify = pd.read_sql("SELECT * FROM clean_dataset LIMIT 5", engine)
display(df_verify)

,date,platform,product_category,model_name,condition,launch_price_usd,launch_price_inr,current_price_usd,current_price_inr,discount_pct,sale_event,stock_status,rating,reviews_count,year,month
0,2020-09-19,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,435.81,43322.4,-1.6,No Sale Event,In Stock,4.7,40,2020,9
1,2020-09-20,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.49,42320.4,-1.7,No Sale Event,Out of Stock,4.6,84,2020,9
2,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,422.73,40879.4,1.5,No Sale Event,In Stock,4.4,110,2020,9
3,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,425.00,42008.7,0.9,No Sale Event,In Stock,4.8,111,2020,9
4,2020-09-24,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.22,41984.3,-1.7,No Sale Event,In Stock,4.7,35,2020,9


In [18]:
original_count = pd.read_sql("SELECT COUNT(*) AS cnt FROM clean_dataset", engine)
print(f"✓ Rows in MySQL  : {original_count['cnt'][0]}")
print(f"✓ Rows in df     : {len(df)}")
print(f"✓ Match          : {original_count['cnt'][0] == len(df)}")

✓ Rows in MySQL  : 80000
✓ Rows in df     : 80000
✓ Match          : True
